# Category Learning

**Track:** Learning | **Function:** Concept formation & classification

**What it measures:** Can the model learn to classify items into categories based on hidden feature rules?

Each item presents objects described by features (shape, color, size, pattern) with category labels. The model must induce the classification rule and categorize new objects.

**Pre/post learning format:**
- *Baseline:* Classify directly from labeled examples
- *Post-learning:* Analyze examples, write notes about the classification rule, then classify with notes

In [ ]:
import kaggle_benchmarks as kbench
import pandas as pd
import random
import string
import re

print('Available models:', list(kbench.llms.keys()))
print('Default model:', kbench.llm)

In [ ]:
def strip_thinking(text: str) -> str:
    """Remove <think>...</think> blocks from reasoning model output."""
    if '</think>' in text:
        return text.split('</think>', 1)[1].strip()
    return text.strip()


def extract_short_answer(response: str) -> str:
    """Extract a short answer from model response (last short line)."""
    text = strip_thinking(response)
    text = re.sub(r'[`*_]', '', text)
    lines = [l.strip() for l in text.strip().split('\n') if l.strip()]
    if not lines:
        return ''
    # Prefer last short line (skip explanations)
    for line in reversed(lines):
        clean = line.strip('"\'\'\u201c\u201d').rstrip('.!?')
        if len(clean) <= 30:
            return clean.lower().strip()
    return lines[-1].strip('"\'\'\u201c\u201d').rstrip('.!?').lower().strip()

In [ ]:
# === Feature-based classification rules ===

SHAPES = ['circle', 'square', 'triangle', 'diamond', 'hexagon', 'star']
COLORS = ['red', 'blue', 'green', 'yellow', 'purple', 'orange']
SIZES = ['tiny', 'small', 'medium', 'large', 'huge']
PATTERNS = ['solid', 'striped', 'dotted', 'checkered', 'plaid']


def make_item(rng):
    """Generate a random item with features."""
    return {
        'shape': rng.choice(SHAPES),
        'color': rng.choice(COLORS),
        'size': rng.choice(SIZES),
        'pattern': rng.choice(PATTERNS),
    }

def item_str(item):
    return f"{item['size']} {item['color']} {item['pattern']} {item['shape']}"


# --- Easy: single-feature rules ---
def rule_single_feature(seed):
    rng = random.Random(seed)
    feature = rng.choice(['shape', 'color', 'size', 'pattern'])
    pool = {'shape': SHAPES, 'color': COLORS, 'size': SIZES, 'pattern': PATTERNS}[feature]
    split = rng.sample(pool, len(pool)//2)
    def classify(item):
        return 'A' if item[feature] in split else 'B'
    desc = f'{feature} in {split} -> A, else B'
    return classify, desc


# --- Medium: two-feature AND rule ---
def rule_two_feature(seed):
    rng = random.Random(seed)
    f1, f2 = rng.sample(['shape','color','size','pattern'], 2)
    pool1 = {'shape':SHAPES,'color':COLORS,'size':SIZES,'pattern':PATTERNS}[f1]
    pool2 = {'shape':SHAPES,'color':COLORS,'size':SIZES,'pattern':PATTERNS}[f2]
    vals1 = set(rng.sample(pool1, max(1, len(pool1)//2)))
    vals2 = set(rng.sample(pool2, max(1, len(pool2)//2)))
    def classify(item):
        cond1 = item[f1] in vals1
        cond2 = item[f2] in vals2
        if cond1 and cond2: return 'A'
        elif cond1: return 'B'
        else: return 'C'
    desc = f'{f1}+{f2} rule'
    return classify, desc


# --- Hard: conditional/hierarchical rule ---
def rule_conditional(seed):
    rng = random.Random(seed)
    f1, f2, f3 = rng.sample(['shape','color','size','pattern'], 3)
    pool1 = {'shape':SHAPES,'color':COLORS,'size':SIZES,'pattern':PATTERNS}[f1]
    pool2 = {'shape':SHAPES,'color':COLORS,'size':SIZES,'pattern':PATTERNS}[f2]
    pool3 = {'shape':SHAPES,'color':COLORS,'size':SIZES,'pattern':PATTERNS}[f3]
    gate_vals = set(rng.sample(pool1, max(1, len(pool1)//2)))
    branch_a = set(rng.sample(pool2, max(1, len(pool2)//2)))
    branch_b = set(rng.sample(pool3, max(1, len(pool3)//2)))
    def classify(item):
        if item[f1] in gate_vals:
            return 'A' if item[f2] in branch_a else 'B'
        else:
            return 'C' if item[f3] in branch_b else 'D'
    desc = f'if {f1} in gate -> {f2} split, else -> {f3} split'
    return classify, desc


RULE_FAMILIES = {
    'easy': rule_single_feature,
    'medium': rule_two_feature,
    'hard': rule_conditional,
}

# Sanity check
clf, desc = rule_conditional(42)
rng = random.Random(99)
for _ in range(3):
    it = make_item(rng)
    print(f'  {item_str(it)} -> {clf(it)}')
print(f'Rule: {desc}')

In [ ]:
N_EX = {'easy': 8, 'medium': 6, 'hard': 4}  # labeled examples shown
N_TEST = 3  # test items per rule instance
SEEDS = 15
rows = []
tid = 0

for diff in ['easy', 'medium', 'hard']:
    n_ex = N_EX[diff]
    rule_fn = RULE_FAMILIES[diff]
    for seed in range(SEEDS):
        clf, rule_desc = rule_fn(seed * 100)
        rng = random.Random(seed * 11 + 7)

        # Generate items ensuring category diversity
        all_items = [make_item(rng) for _ in range(200)]
        # Label them
        labeled = [(it, clf(it)) for it in all_items]
        # Ensure we have multiple categories in examples
        cats = set()
        chosen_ex = []
        for it, cat in labeled:
            if len(chosen_ex) < n_ex:
                chosen_ex.append((it, cat))
                cats.add(cat)
        # Pick test items not in examples
        remaining = [(it, cat) for it, cat in labeled[n_ex:] if it not in [e[0] for e in chosen_ex]]
        test_items = remaining[:N_TEST]

        ex_text = '\n'.join(f'{item_str(it)} -> Category {cat}' for it, cat in chosen_ex)

        for it, expected_cat in test_items:
            rows.append({
                'task_id': tid, 'seed': seed, 'difficulty': diff,
                'rule_desc': rule_desc, 'n_examples': n_ex,
                'examples_text': ex_text,
                'test_item': item_str(it), 'expected': expected_cat,
            })
            tid += 1

dataset = pd.DataFrame(rows)
print(f'Dataset: {len(dataset)} items')
print(dataset['difficulty'].value_counts().to_string())

In [ ]:
BASELINE_P = '''Objects have been classified into categories based on a hidden rule.
Study the labeled examples, then classify the test object.

Labeled examples:
{examples}

What category does this object belong to: {test_item}

Reply with ONLY the category letter (e.g. A, B, C, or D). Nothing else.'''

STUDY_P = '''Objects have been classified into categories based on a hidden rule.
Analyze these labeled examples to discover the classification rule.

Labeled examples:
{examples}

For each category, identify which features determine membership.
Write down the EXACT rule. Test it against every example to verify.'''

APPLY_P = '''You previously analyzed a classification rule and wrote these notes:

--- YOUR NOTES ---
{notes}
--- END NOTES ---

Original labeled examples:
{examples}

What category does this object belong to: {test_item}

Reply with ONLY the category letter (e.g. A, B, C, or D). Nothing else.'''

def extract_category(response):
    text = strip_thinking(response)
    text = re.sub(r'[`*_]', '', text)
    lines = [l.strip() for l in text.strip().split('\n') if l.strip()]
    for line in reversed(lines):
        # Look for standalone category letter
        m = re.search(r'\b([A-D])\b', line)
        if m and len(line) < 30:
            return m.group(1)
    # Fallback: first capital letter A-D anywhere
    for line in reversed(lines):
        m = re.search(r'([A-D])', line)
        if m:
            return m.group(1)
    return ''

## Task 1: Baseline (classify directly from examples)

In [ ]:
@kbench.task(name='category_baseline',
             description='Classify objects using hidden feature rule from examples alone')
def category_baseline(llm, examples_text:str, test_item:str,
                      expected:str, difficulty:str, seed:int,
                      rule_desc:str, n_examples:int) -> bool:
    resp = llm.prompt(BASELINE_P.format(examples=examples_text, test_item=test_item))
    return extract_category(resp) == expected

baseline_runs = category_baseline.evaluate(
    llm=[kbench.llm], evaluation_data=dataset.set_index('task_id'),
    n_jobs=1, timeout=3600, max_attempts=2)
baseline_df = baseline_runs.as_dataframe()
print(f'Baseline done. Accuracy: {baseline_df["result"].mean():.1%}' if 'result' in baseline_df.columns else 'Baseline done.')

## Task 2: Post-learning (analyze rule + classify with notes)

In [ ]:
@kbench.task(name='category_learning',
             description='Analyze classification rule, generate notes, then classify with notes')
def category_learning(llm, examples_text:str, test_item:str,
                      expected:str, difficulty:str, seed:int,
                      rule_desc:str, n_examples:int) -> bool:
    notes = strip_thinking(llm.prompt(STUDY_P.format(examples=examples_text)))
    resp = llm.prompt(APPLY_P.format(notes=notes, examples=examples_text, test_item=test_item))
    return extract_category(resp) == expected

learning_runs = category_learning.evaluate(
    llm=[kbench.llm], evaluation_data=dataset.set_index('task_id'),
    n_jobs=1, timeout=3600, max_attempts=2)
learning_df = learning_runs.as_dataframe()
print(f'Learning done. Accuracy: {learning_df["result"].mean():.1%}' if 'result' in learning_df.columns else 'Learning done.')

## Results

In [ ]:
# === Combine results ===
analysis = dataset.copy()

def safe_map_results(df, col_name):
    if 'result' in df.columns:
        result_map = dict(zip(df.index if 'task_id' not in df.columns else df['task_id'], df['result']))
        analysis[col_name] = analysis['task_id'].map(result_map).fillna(False).astype(bool)
    else:
        analysis[col_name] = False

safe_map_results(baseline_df, 'baseline_correct')
safe_map_results(learning_df, 'learning_correct')

baseline_acc = analysis['baseline_correct'].mean()
learning_acc = analysis['learning_correct'].mean()
learning_gain = learning_acc - baseline_acc
room = 1.0 - baseline_acc
efficiency = learning_gain / room if room > 0 else 0.0

print('=' * 60)
print('CATEGORY LEARNING — LEARNING RESULTS')
print('=' * 60)
print(f'Baseline accuracy:      {baseline_acc:.1%}')
print(f'Post-learning accuracy: {learning_acc:.1%}')
print(f'Learning gain:          {learning_gain:+.1%}')
print(f'Learning efficiency:    {efficiency:.1%} (of possible improvement)')
print()

print('By Difficulty:')
print('-' * 60)
for diff in ['easy', 'medium', 'hard']:
    subset = analysis[analysis['difficulty'] == diff]
    if len(subset) == 0:
        continue
    b = subset['baseline_correct'].mean()
    l = subset['learning_correct'].mean()
    g = l - b
    print(f'  {diff:8s}  baseline={b:.1%}  learned={l:.1%}  gain={g:+.1%}  (n={len(subset)})')

print()
helped = analysis[~analysis['baseline_correct'] & analysis['learning_correct']]
hurt = analysis[analysis['baseline_correct'] & ~analysis['learning_correct']]
print(f'Learning helped (wrong->right): {len(helped)}')
print(f'Learning hurt (right->wrong):   {len(hurt)}')
print(f'Net learning impact:            {len(helped) - len(hurt):+d} items')


In [ ]:
!pip install -q matplotlib
import matplotlib.pyplot as plt

difficulties = ['easy', 'medium', 'hard']
baseline_scores = [analysis[analysis['difficulty'] == d]['baseline_correct'].mean() for d in difficulties if len(analysis[analysis['difficulty'] == d]) > 0]
learning_scores = [analysis[analysis['difficulty'] == d]['learning_correct'].mean() for d in difficulties if len(analysis[analysis['difficulty'] == d]) > 0]
diff_labels = [d for d in difficulties if len(analysis[analysis['difficulty'] == d]) > 0]

x = range(len(diff_labels))
width = 0.35

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax1 = axes[0]
ax1.bar([i - width/2 for i in x], baseline_scores, width, label='Baseline', color='#4C72B0')
ax1.bar([i + width/2 for i in x], learning_scores, width, label='Post-Learning', color='#55A868')
ax1.set_xlabel('Difficulty')
ax1.set_ylabel('Accuracy')
ax1.set_title('Baseline vs Post-Learning Accuracy')
ax1.set_xticks(list(x))
ax1.set_xticklabels(diff_labels)
ax1.legend()
ax1.set_ylim(0, 1.05)

ax2 = axes[1]
gains = [l - b for b, l in zip(baseline_scores, learning_scores)]
colors = ['#55A868' if g >= 0 else '#C44E52' for g in gains]
ax2.bar(diff_labels, gains, color=colors)
ax2.set_xlabel('Difficulty')
ax2.set_ylabel('Learning Gain')
ax2.set_title('Learning Gain by Difficulty')
ax2.axhline(y=0, color='gray', linestyle='--', linewidth=0.8)

plt.tight_layout()
plt.savefig('category_learning_results.png', dpi=150, bbox_inches='tight')
plt.show()
